In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [3]:
# GoEmotions has 28 emotions trained on Reddit data
emotion_model = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    truncation=True
)

# Quick test
test = emotion_model("I'm so excited for the Super Bowl!")
print(f"Number of labels: {len(test[0])}")
print(f"All labels: {[item['label'] for item in test[0]]}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2430.34it/s]


Number of labels: 28
All labels: ['excitement', 'joy', 'curiosity', 'neutral', 'approval', 'surprise', 'admiration', 'amusement', 'love', 'desire', 'gratitude', 'confusion', 'optimism', 'fear', 'annoyance', 'caring', 'nervousness', 'realization', 'disapproval', 'disgust', 'anger', 'sadness', 'relief', 'disappointment', 'embarrassment', 'pride', 'remorse', 'grief']


In [4]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:2000] for t in texts]

print(f"Running GoEmotions model on {len(texts)} posts...")
results = emotion_model(texts, batch_size=16)
print(f"Got {len(results)} results")

Running GoEmotions model on 797 posts...
Got 797 results


In [5]:
# Get the label list from the first result
emotion_labels = sorted([item["label"] for item in results[0]])
emo_cols = [f"emo_{c}" for c in emotion_labels]
print(f"Labels: {emotion_labels}")
print(f"Total: {len(emotion_labels)} emotions")

Labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'neutral', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise']
Total: 28 emotions


In [6]:
# Drop existing emotion columns if they exist (safe re-run)
df = df.drop(columns=[c for c in emo_cols + ["dominant_emotion", "dominant_emotion_score"] if c in df.columns])

# Build score dataframe
rows = [{item["label"]: item["score"] for item in res} for res in results]
emo_df = pd.DataFrame(rows)[emotion_labels]
emo_df.columns = emo_cols
emo_df.index = df.index[df["has_text"]]
df = df.join(emo_df)

# Compute dominant emotion (only on rows with text)
df["dominant_emotion"] = None
df["dominant_emotion_score"] = None
df.loc[df["has_text"], "dominant_emotion"] = (
    df.loc[df["has_text"], emo_cols].idxmax(axis=1).str.replace("emo_", "")
)
df.loc[df["has_text"], "dominant_emotion_score"] = df.loc[df["has_text"], emo_cols].max(axis=1)

print(df[["student_id", "text_source", "dominant_emotion", "dominant_emotion_score"]].head())

  student_id         text_source dominant_emotion dominant_emotion_score
0        1_A  caption+transcript          neutral                0.76296
1        1_A  caption+transcript           caring               0.857797
2        1_A        caption_only        curiosity               0.534555
3        2_A        caption_only          neutral               0.965802
4        2_A        caption_only          neutral               0.915798


In [7]:
print("=== Overall dominant emotion counts ===")
print(df["dominant_emotion"].value_counts(dropna=False))
print()
print("=== Top 10 emotions by mean score across all posts ===")
print(df[emo_cols].mean().sort_values(ascending=False).head(10))

=== Overall dominant emotion counts ===
dominant_emotion
neutral           503
None              108
curiosity          44
gratitude          39
excitement         32
approval           28
admiration         26
love               23
amusement          16
caring             14
optimism           12
joy                10
sadness            10
confusion          10
disappointment      6
desire              5
fear                4
annoyance           3
surprise            3
realization         3
disapproval         2
remorse             2
embarrassment       1
nervousness         1
Name: count, dtype: int64

=== Top 10 emotions by mean score across all posts ===
emo_neutral       0.546626
emo_curiosity     0.082407
emo_approval      0.068002
emo_admiration    0.053710
emo_excitement    0.049156
emo_confusion     0.048387
emo_gratitude     0.046657
emo_love          0.033431
emo_joy           0.030209
emo_optimism      0.023713
dtype: float64


In [8]:
# Only show emotions that have at least 5 posts (28 is a lot of categories)
common_emotions = df["dominant_emotion"].value_counts()
common_emotions = common_emotions[common_emotions >= 5].index.tolist()

for emo in common_emotions[:10]:  # Top 10 most common emotions
    subset = df[df["dominant_emotion"] == emo]
    print(f"\n=== {emo.upper()} ({len(subset)} posts) ===")
    top = subset.nlargest(2, f"emo_{emo}")
    for _, r in top.iterrows():
        text_preview = r['text_for_analysis'][:180].replace('\n', ' | ')
        print(f"  [{r[f'emo_{emo}']:.2f}] [{r['text_source']}] {text_preview}")


=== NEUTRAL (503 posts) ===
  [0.97] [caption_only] #foryoupage #xzybca #real #her
  [0.97] [caption_only] #crete  #travel  #greece  #fy  #cretanbluerestaurant

=== CURIOSITY (44 posts) ===
  [0.75] [caption_only] "First official look at Dwayne Johnson as Maui in the live-action Moana remake. The film will also star Catherine Lagaaia as Moana, John Tui as Chief Tui, Frankie Adams as Sina
  [0.74] [caption_only] How would you feel if the Jets drafted Rueben Bain Jr. with the second overall pick?

=== GRATITUDE (39 posts) ===
  [0.99] [caption_only] THANK YOU THANK YOU THANK YOU!!!!!!! SOTREMENDOUSLY GRATEFUL!!!!!!
  [0.99] [caption_only] Im incredibly grateful for this opportunity and for everyone who has supported me throughout my time at Syracuse University and beyond. From professors and mentors to friends and 

=== EXCITEMENT (32 posts) ===
  [0.82] [caption_only] Im excited to share that Ill be joining Glassnote Entertainment Group as a Summer 2026 intern!
  [0.81] [caption

In [9]:
output_path = "../Outputs/emotion_goemotions_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/emotion_goemotions_905.csv


# Emotion Model 2: GoEmotions — Conclusion

**Model:** `SamLowe/roberta-base-go_emotions`
**Output labels:** 28 emotions including admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral
**Input used:** caption + transcript combined where available, caption only otherwise
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

We ran every post through a second emotion model. This one is trained on Reddit data and outputs 28 different emotion categories instead of 7. The same post gets 28 different probability scores, and the highest one becomes the dominant emotion.

## What we found overall

Out of 797 usable posts:
- 503 came out as neutral (63%)
- 44 as curiosity
- 39 as gratitude
- 32 as excitement
- 28 as approval
- 26 as admiration
- 23 as love
- The rest spread across the remaining 21 categories with smaller counts

## Why this model is better than the first one

The first model (DistilRoBERTa with 7 emotions) had a clear problem: it kept tagging entertainment content as anger or fear. A wholesome video game post got tagged 92% angry. A Chanel shopping vlog got tagged 91% angry. An Avatar cartoon clip that literally said "lightning is not fueled by rage" got tagged 99% angry.

GoEmotions doesn't have that problem. Out of 797 posts:
- DistilRoBERTa flagged 146 as fear and 46 as anger (192 posts total)
- GoEmotions flagged 4 as fear and 0 as anger (4 posts total)

The reason is simple. DistilRoBERTa only had 7 categories to choose from. When it saw words like "deadly" or "rage" in entertainment content, anger was the closest fit, so it picked anger. GoEmotions has 28 categories, so it can pick something more accurate like curiosity, realization, or amusement instead.

When we spot-checked the GoEmotions results, the labels actually fit the posts:
- Gratitude posts: "THANK YOU THANK YOU THANK YOU" and "I'm incredibly grateful for this opportunity"
- Excitement posts: "I'm excited to share I'll be joining Glassnote"
- Caring posts: "I pray that the pain you've been carrying begins to ease"
- Amusement posts: "HAHAHAHAHAH he just had orange Gatorade"
- Optimism posts: "I'm looking forward to growing academically"

These labels are right. The model is actually understanding what the posts are about, not just matching keywords.

## But here's the catch: the neutral problem

63% of posts came out as neutral. That's a lot. The first model only had 35% neutral.

What's happening: GoEmotions is much more conservative. It only fires a specific emotion when it's really confident. If the text isn't clearly emotional, it defaults to neutral and moves on. The first model was the opposite, trying to assign some emotion to almost every post even when it had no good evidence.

This creates a tradeoff:
- DistilRoBERTa is over-eager. It labels too many posts as emotional, often wrong.
- GoEmotions is too cautious. It labels too few posts as emotional, missing some real emotional content.

Both have problems. They just have opposite problems.

## What this means for the project

Neither model is right on its own. Looking at the same 797 posts:
- One model says 28% are angry, fearful, or disgusted
- The other model says less than 1% are
- Both can't be true

This is exactly the evidence we needed. The two models tell wildly different stories about the same content. We cannot trust any single transformer model's labels for individual posts.

There are three ways forward:
1. Run multiple models and use only labels where multiple models agree
2. Use averaging across models to get a more stable signal
3. Move to an LLM (like Claude or GPT) that can actually understand context instead of pattern-matching on words

## Bottom line

GoEmotions is more accurate on the posts it does label. When it says a post is gratitude or excitement, it's usually right. But it labels too few posts as anything beyond neutral, so we lose coverage in exchange for accuracy.

DistilRoBERTa is the opposite: high coverage, low accuracy on individual posts.

For aggregate student-level analysis (e.g., "what kinds of content does student X consume?"), this comparison is useful. For per-post labels, we now have strong evidence that no single transformer model is reliable enough on social media content.